In [1]:
# IMPORT LIBRARIES

import os
import json
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    precision_recall_curve,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")

In [2]:
# GLOBAL CONFIGURATION

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

IMAGE_HEIGHT = 128
IMAGE_WIDTH = 128

BATCH_SIZE = 32

EPOCHS = 20

LEARNING_RATE = 1e-3

INPUT_SHAPE = (IMAGE_HEIGHT, IMAGE_WIDTH, 3)

print("=" * 60)
print("TensorFlow Version :", tf.__version__)
print("Random Seed        :", SEED)
print("Image Size         :", INPUT_SHAPE)
print("Batch Size         :", BATCH_SIZE)
print("Epochs             :", EPOCHS)
print("=" * 60)

TensorFlow Version : 2.20.0
Random Seed        : 42
Image Size         : (128, 128, 3)
Batch Size         : 32
Epochs             : 20


In [3]:
# GPU CHECK

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print(f"GPU Available : {len(gpus)}")

    for gpu in gpus:
        print(gpu)

else:
    print("No GPU detected. Training will use CPU.")

GPU Available : 1
PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [4]:
import shutil

SOURCE_IMAGES = "/content/drive/MyDrive/skin_cancer_vs_Benign_tumor_cnn_model_personal/dataset"
DEST_IMAGES = "/content/dataset"

if not os.path.exists(DEST_IMAGES):
    print("Copying images to local colab SSD...")
    shutil.copytree(SOURCE_IMAGES, DEST_IMAGES)
    print("Done!")
else:
    print("Images already copied.")

Copying images to local colab SSD...
Done!


In [5]:
# LOAD PREPROCESSED DATA
from pathlib import Path

DATASET_SPLIT = Path( "/content/drive/MyDrive/skin_cancer_vs_Benign_tumor_cnn_model_personal/dataset_split")

train_df = pd.read_csv(f"{DATASET_SPLIT}/train.csv")
val_df   = pd.read_csv(f"{DATASET_SPLIT}/val.csv")
test_df  = pd.read_csv(f"{DATASET_SPLIT}/test.csv")

print("Training :", len(train_df))
print("Validation :", len(val_df))
print("Testing :", len(test_df))

print()

print("Training Class Distribution")
print(train_df["binary_label"].value_counts())

print()

with open(f"{DATASET_SPLIT}/class_weights.json","r") as f:
    class_weights=json.load(f)

class_weights={int(k):float(v) for k,v in class_weights.items()}

print()

print("Class Weights")
print(class_weights)

Training : 7010
Validation : 1500
Testing : 1505

Training Class Distribution
binary_label
0    5641
1    1369
Name: count, dtype: int64


Class Weights
{0: 0.6213437333806062, 1: 2.560262965668371}


In [6]:
# CONVERT KERAS BINARY LABEL TO STRING

for df in [train_df, val_df, test_df]:
    df["binary_label_str"] = df["binary_label_str"].astype(str)

In [7]:
from pathlib import Path

# Local dataset location (copied from Google Drive)
IMAGE_DATASET = Path("/content/dataset")

part1 = IMAGE_DATASET / "HAM10000_images_part_1"
part2 = IMAGE_DATASET / "HAM10000_images_part_2"

image_lookup = {}

for folder in [part1, part2]:
    for img in folder.glob("*.jpg"):
        image_lookup[img.name] = str(img)

for df in [train_df, val_df, test_df]:
    df["filepath"] = df["image_id"].map(image_lookup)

print("Filepaths updated to local dataset.")

Filepaths updated to local dataset.


In [8]:
# IMAGE DATA GENERATORS

from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1/255,
    rotation_range=20,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10,
    horizontal_flip=True
)

eval_datagen = ImageDataGenerator(
    rescale=1/255
)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col="filepath",
    y_col="binary_label_str",
    directory=None,
    target_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=True,
    seed=SEED
)

val_generator = eval_datagen.flow_from_dataframe(
    dataframe=val_df,
    directory=None,
    x_col="filepath",
    y_col="binary_label_str",
    target_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

test_generator = eval_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=None,
    x_col="filepath",
    y_col="binary_label_str",
    target_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

print()
print("Class Indices")
print(train_generator.class_indices)

Found 7010 validated image filenames belonging to 2 classes.
Found 1500 validated image filenames belonging to 2 classes.
Found 1505 validated image filenames belonging to 2 classes.

Class Indices
{'0': 0, '1': 1}


In [9]:
# BUILD CUSTOM CNN

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    BatchNormalization,
    GlobalAveragePooling2D,
    Dense,
    Dropout
)
from tensorflow.keras.regularizers import l2


def build_custom_cnn(input_shape):
    """
    A Four-block CNN built entirely from scratch.
    Filters double each block (32 >> 64 >> 128 >> 256) following a standard design pattern.
    """

    model = Sequential(name="SkinCancer_CustomCNN")

    # Input Layer
    model.add(Input(shape=input_shape))


    # Block 1 : 32 filters
    model.add(
        Conv2D(
            filters=32,
            kernel_size=(3,3),
            padding="same",
            activation="relu",
            kernel_regularizer=l2(1e-4)
        )
    )
    model.add(BatchNormalization())
    model.add(
        Conv2D(
            filters=32,
            kernel_size=(3,3),
            activation="relu",
            padding="same",
            kernel_regularizer=l2(1e-4)
        )
    )
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2,2)))
    model.add(Dropout(0.25))


    # Block 2 : 64 filters
    model.add(
        Conv2D(
            filters=64,
            kernel_size=(3,3),
            activation="relu",
            padding="same",
            kernel_regularizer=l2(1e-4)
        )
    )
    model.add(BatchNormalization())
    model.add(
        Conv2D(
            filters=64,
            kernel_size=(3,3),
            activation="relu",
            padding="same",
            kernel_regularizer=l2(1e-4)
        )
    )
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2,2)))
    model.add(Dropout(0.30))


    # Block 3 : 128 filters
    model.add(
        Conv2D(
            filters=128,
            kernel_size=(3,3),
            activation="relu",
            padding="same",
            kernel_regularizer=l2(1e-4)
        )
    )
    model.add(BatchNormalization())
    model.add(
        Conv2D(
            filters=128,
            kernel_size=(3,3),
            activation="relu",
            padding="same",
            kernel_regularizer=l2(1e-4)
        )
    )
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2,2)))
    model.add(Dropout(0.35))


    # Block 4 : 256 filters
    model.add(
        Conv2D(
            filters=256,
            kernel_size=(3,3),
            activation="relu",
            padding="same",
            kernel_regularizer=l2(1e-4)
        )
    )
    model.add(BatchNormalization())
    model.add(
        Conv2D(
            filters=256,
            kernel_size=(3,3),
            activation="relu",
            padding="same",
            kernel_regularizer=l2(1e-4)
        )
    )
    model.add(BatchNormalization())

    model.add(MaxPooling2D(pool_size=(2,2)))
    model.add(Dropout(0.40))

    # Classification Head
    model.add(GlobalAveragePooling2D())
    model.add(Dense(128, activation="relu", kernel_regularizer=l2(1e-4)))
    model.add(Dropout(0.50))
    model.add(Dense(1, activation="sigmoid"))

    return model


cnn_model = build_custom_cnn(INPUT_SHAPE)

cnn_model.summary()

Model: "SkinCancer_CustomCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 64, 64, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 16, 16, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 16, 16, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 16, 16, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 1,209,121 (4.61 MB)

 Trainable params: 1,207,201 (4.61 MB)

 Non-trainable params: 1,920 (7.50 KB)

In [10]:
# COMPILE MODEL & CONFIGURE CALLBACKS

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)

# Create folder for saved models
os.makedirs("models", exist_ok=True)


# Compile Model
cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)


# Callbacks
callbacks = [

    # Save the best model
    ModelCheckpoint(
        filepath="/content/drive/MyDrive/skin_cancer_vs_Benign_tumor_cnn_model_personal/models/custom_cnn_best.keras",
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    # Stop training if validation performance stops improving
    EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),

    # Reduce learning rate when learning plateaus
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )

]

print("=" * 40)
print("Model compiled successfully.")
print("Callbacks configured.")
print("=" * 40)

Model compiled successfully.
Callbacks configured.


In [11]:
# TRAIN CUSTOM CNN

cnn_history = cnn_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining Complete!")

Epoch 1/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 516ms/step - accuracy: 0.5960 - auc: 0.6751 - loss: 0.8531 - precision: 0.2797 - recall: 0.6953
Epoch 1: val_auc improved from None to 0.68754, saving model to /content/drive/MyDrive/skin_cancer_vs_Benign_tumor_cnn_model_personal/models/custom_cnn_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/skin_cancer_vs_Benign_tumor_cnn_model_personal/models/custom_cnn_best.keras
220/220 ━━━━━━━━━━━━━━━━━━━━ 141s 580ms/step - accuracy: 0.6033 - auc: 0.7227 - loss: 0.7301 - precision: 0.3009 - recall: 0.7794 - val_accuracy: 0.7920 - val_auc: 0.6875 - val_loss: 0.7263 - val_precision: 0.4380 - val_recall: 0.1785 - learning_rate: 0.0010
Epoch 2/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 430ms/step - accuracy: 0.6142 - auc: 0.7675 - loss: 0.6556 - precision: 0.3207 - recall: 0.8614
Epoch 2: val_auc did not improve from 0.68754
220/220 ━━━━━━━━━━━━━━━━━━━━ 106s 482ms/step - accuracy: 0.6177 - auc: 0.7627 - loss: 0.6547 - precision: 0.3196 - recal